## CLS Token
ViT 通常在 Patch Token 前加入一个可学习的 CLS Token：
$$
X=[x_{\text{cls}},x_1,x_2,\dots,x_{196}]
$$
因此 Token 数量从 196 变为：
$$
197
$$
输入形状：
$$
X\in\mathbb{R}^{197\times768}
$$
经过 Transformer Encoder 后，取 CLS Token：
$$
h_{\text{cls}}\in\mathbb{R}^{768}
$$
用于图像分类：
$$
y=\text{Linear}(h_{\text{cls}})
$$
这和 BERT 使用 [CLS] 表示整个句子非常相似。
## 位置编码
Transformer 本身不知道 Patch 的空间位置。
例如，它无法区分：
左上角的 Patch
右下角的 Patch
因此需要加入位置编码：
$$
Z_0=[x_{\text{cls}};x_1E;x_2E;\dots;x_NE]+E_{\text{pos}}
$$
其中：
$$
E_{\text{pos}}\in\mathbb{R}^{(N+1)\times D}
$$
对于 224×224、Patch 16：
$$
E_{\text{pos}}\in\mathbb{R}^{197\times768}
$$
ViT 通常使用可学习位置编码：
```python
self.position_embedding = nn.Parameter(
    torch.randn(1, 197, 768)
)
```


``` text
[B, 3, 224, 224]
        ↓ Conv2d
[B, 768, 14, 14]
        ↓ Flatten
[B, 768, 196]
        ↓ Transpose
[B, 196, 768]
        ↓ 添加 CLS
[B, 197, 768]
        ↓ 添加位置编码
[B, 197, 768]
```

In [ ]:
class ViTEmbedding(nn.Module):
    def __init__(
        self,
        image_size: int = 224,
        patch_size: int = 16,
        embed_dim: int = 768,
    ):
        super().__init__()

        self.patch_embedding = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
        )

        num_patches = (image_size // patch_size) ** 2

        self.cls_token = nn.Parameter(
            torch.randn(1, 1, embed_dim)
        )

        self.position_embedding = nn.Parameter(
            torch.randn(1, num_patches + 1, embed_dim)
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        batch_size = images.shape[0]

        patch_tokens = self.patch_embedding(images)
        # [B, 196, 768]

        cls_tokens = self.cls_token.expand(
            batch_size, -1, -1
        )
        # [B, 1, 768]

        tokens = torch.cat(
            [cls_tokens, patch_tokens],
            dim=1,
        )
        # [B, 197, 768]

        tokens = tokens + self.position_embedding

        return tokens

## 位置编码与Transformer不同

GPT 位置编码是可学习位置编码
Transformer 位置编码是正/余弦位置编码

- 最重要的是：GPT中使用可学习位置编码的效果更好，但是存在文本长度限制
- Transformer 在意的是外推能力，GPT中在意的是拟合效果


``` text
2017 原版 Transformer   --->  2018~2020 GPT 时代      --->  2023+ 现代大模型 (LLaMA等)
使用了“静态正弦公式”           切换为“可学习位置编码”         重新回归“基于三角函数的相对编码”
(追求理论外推，但实际有限)      (数据量大，追求拟合与简单)       (RoPE 旋转位置编码，兼顾拟合与真正外推)
```

>现代 LLM（如 LLaMA / Qwen） 发现可学习位置编码在面对扩展上下文（如从 4k 到 128k）时完全束手无策，于是结合了正弦公式的“三角函数旋转”与“相对位置”思想，发明了 **RoPE**（旋转位置编码），终于真正实现了既有强拟合力，又能通过插值轻松外推的能力。